# Session 1: the line that misses least
**A. Real-world problem.** A farmer has ten seasons of records: how much rain fell, and how much
grain the field gave. Next season's rain forecast is known. What yield should the farmer expect?
We draw a straight line through the dots and read the answer off it. But many lines are possible,
so we need a fair rule for "best". All data here are made up for teaching.

**Plan.** (1) store the dots, (2) draw any line y = a + b x, (3) measure each miss (a residual),
(4) add up the squared misses, (5) find the line with the smallest total.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# rain in hundreds of mm per season (3.0 means 300 mm); yield in tonnes per hectare
RAIN = np.array([3.0, 3.8, 4.5, 5.2, 5.8, 6.5, 7.2, 7.9, 8.6, 9.2])
YIELD = np.array([2.0, 2.3, 2.4, 2.5, 2.8, 2.9, 3.4, 4.1, 3.8, 3.9])

## Step 1: a line and its misses
A line is y = a + b x. Here `a` is the yield with no rain (the intercept, t/ha) and `b` is the
extra yield for each extra 100 mm of rain (the slope, t/ha per 100 mm). A *residual* is
observed yield minus the line's prediction: positive when the dot is above the line.

In [ ]:
def predict(a, b, x):
    """Line prediction a + b x."""
    return a + b * np.asarray(x, float)


def residuals(a, b, x, y):
    """Observed minus predicted (the vertical miss of each dot)."""
    return np.asarray(y, float) - predict(a, b, x)


def sse(a, b, x, y):
    """Sum of squared residuals: the total squared miss."""
    return float(np.sum(residuals(a, b, x, y) ** 2))

## Step 2: why squares?
Misses above and below the line would cancel if we just added them. Squaring makes every miss
positive and makes one big miss count more than several small ones. The best line is the one
with the smallest total squared miss. Calculus gives it in closed form (Session 2 derives it):

    b = sum((x - xbar) * (y - ybar)) / sum((x - xbar)^2)        a = ybar - b * xbar

In [ ]:
def best_line(x, y):
    """Least-squares intercept and slope from the closed form."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    xm, ym = x.mean(), y.mean()
    b = np.sum((x - xm) * (y - ym)) / np.sum((x - xm) ** 2)
    return float(ym - b * xm), float(b)


def sse_for_slope(b, x, y):
    """Smallest total squared miss over all intercepts when the slope is fixed at b."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    a = y.mean() - b * x.mean()
    return sse(a, b, x, y)


def scan_best_slope(x, y, lo=0.0, hi=0.8, n=801):
    """Brute force: try many slopes, return the slope with the smallest squared miss."""
    slopes = np.linspace(lo, hi, n)
    errs = np.array([sse_for_slope(b, x, y) for b in slopes])
    return float(slopes[np.argmin(errs)])

## Step 3: see it
Left: the dots, a guess line and the residuals as vertical bars; the best line has the smallest
total bar-area. Right: total squared miss against slope; the lowest point of the bowl is the
best slope.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    a0, b0 = best_line(RAIN, YIELD)
    print(f"best line: yield = {a0:.3f} + {b0:.4f} * rain   (t/ha, rain in 100 mm)")
    print(f"total squared miss, best line: {sse(a0, b0, RAIN, YIELD):.3f}")
    print(f"total squared miss, guess line 1.0 + 0.30 x: {sse(1.0, 0.30, RAIN, YIELD):.3f}")
    print(f"brute-force slope: {scan_best_slope(RAIN, YIELD):.3f}")
    xs = np.linspace(2.5, 9.7, 50)
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    ax = axs[0]
    ax.scatter(RAIN, YIELD, color="tab:blue", zorder=3, label="seasons")
    ax.plot(xs, predict(1.0, 0.30, xs), color="tab:red", ls="--", label="guess: 1.0 + 0.30 x")
    ax.vlines(RAIN, predict(1.0, 0.30, RAIN), YIELD, color="tab:red", alpha=0.6)
    ax.plot(xs, predict(a0, b0, xs), color="tab:green", label=f"best: {a0:.2f} + {b0:.3f} x")
    ax.set_xlabel("rainfall (hundreds of mm)")
    ax.set_ylabel("yield (t/ha)")
    ax.set_title("Residuals are the vertical bars")
    ax.grid(alpha=0.3)
    ax.legend()
    bs = np.linspace(0, 0.8, 100)
    axs[1].plot(bs, [sse_for_slope(b, RAIN, YIELD) for b in bs])
    axs[1].axvline(b0, color="gray", ls=":")
    axs[1].set_xlabel("slope b (t/ha per 100 mm)")
    axs[1].set_ylabel("total squared miss (t/ha)^2")
    axs[1].set_title("Smallest total squared miss at the best slope")
    axs[1].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "dots_and_line.png", dpi=120)
    print("predicted yield at 600 mm:", round(float(predict(a0, b0, 6.0)), 2), "t/ha")
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the checkpoint.

## Coding tasks
1. **Easy.** Change the guess line in `lesson.py` to 0.5 + 0.40 x and print its total squared miss. Is it better than 1.0 + 0.30 x?
2. **Stretch.** Write `forecast(rain_mm)` that uses the best line and prints the yield for 400, 600 and 800 mm.
3. **Explain.** Explain, in two sentences, why the line that minimises the sum of the absolute residuals could be a different line from the least-squares line.

## Assessment
- **A. Conceptual.** Why do we square the residuals instead of just adding them up? What would happen to the line 'sum of residuals = 0' test?
- **B. Numerical.** Find the least-squares line for the dots (0, 2), (1, 1) and (2, 6), and its total squared miss.
- **C. Graph and simulation interpretation.** In the bowl view, at which slope is the green curve lowest, and what is its value there? Why is the blue curve never below the green one?
- **D. Design and optimisation challenge.** Design: using the best line of the farm data, how much rain gives an expected yield of 3.5 t/ha? Is the answer inside the range of the data? What should you worry about?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    x3, y3 = np.array([1.0, 2.0, 3.0]), np.array([2.0, 2.0, 5.0])
    a3, b3 = L.best_line(x3, y3)
    checks.append(("worked example: three dots (1,2), (2,2), (3,5) give y = 0 + 1.5 x", abs(a3) < 1e-9 and abs(b3 - 1.5) < 1e-9))
    checks.append(("worked example: its residuals are 0.5, -1, 0.5 and total squared miss 1.5",
                   np.allclose(L.residuals(a3, b3, x3, y3), [0.5, -1.0, 0.5]) and abs(L.sse(a3, b3, x3, y3) - 1.5) < 1e-9))
    a, b = L.best_line(L.RAIN, L.YIELD)
    checks.append((f"farm data: best line is 0.887 + 0.344 x (got {a:.3f} + {b:.3f} x)", abs(a - 0.887) < 5e-4 and abs(b - 0.3441) < 5e-4))
    checks.append(("the best line matches np.polyfit", np.allclose([b, a], np.polyfit(L.RAIN, L.YIELD, 1), atol=1e-9)))
    checks.append(("residuals of the best line add up to zero", abs(L.residuals(a, b, L.RAIN, L.YIELD).sum()) < 1e-9))
    checks.append(("best line: total squared miss is 0.378 (t/ha)^2", abs(L.sse(a, b, L.RAIN, L.YIELD) - 0.378) < 5e-4))
    checks.append(("guess line 1.0 + 0.30 x: total squared miss is 0.706 (t/ha)^2", abs(L.sse(1.0, 0.30, L.RAIN, L.YIELD) - 0.706) < 5e-4))
    checks.append(("brute-force scan finds the same slope, 0.344", abs(L.scan_best_slope(L.RAIN, L.YIELD) - b) < 2e-3))
    rng = np.random.default_rng(0)
    worse = all(L.sse(a + da, b + db, L.RAIN, L.YIELD) > L.sse(a, b, L.RAIN, L.YIELD)
                for da, db in rng.normal(0, 0.1, (200, 2)))
    checks.append(("200 random nudges of the best line all make the miss larger", worse))
    checks.append(("forecast: 600 mm of rain gives 2.95 t/ha", abs(float(L.predict(a, b, 6.0)) - 2.95) < 5e-3))
    checks.append(("edge: two dots are fitted exactly (zero miss)", L.sse(*L.best_line([1, 2], [3, 7]), [1, 2], [3, 7]) < 1e-18))
    a2, b2 = L.best_line([0, 1, 2], [2, 1, 6])
    checks.append(("question B: dots (0,2), (1,1), (2,6) give y = 1 + 2 x with total squared miss 6",
                   abs(a2 - 1) < 1e-9 and abs(b2 - 2) < 1e-9 and abs(L.sse(a2, b2, [0, 1, 2], [2, 1, 6]) - 6) < 1e-9))
    checks.append(("any line through (x-bar, y-bar) has residuals adding to zero, whatever the slope",
                   abs(L.residuals(L.YIELD.mean() - 0.1 * L.RAIN.mean(), 0.1, L.RAIN, L.YIELD).sum()) < 1e-9))
    checks.append(("question D: 3.5 t/ha needs rain (3.5 - 0.887) / 0.344 = 7.59, inside the data range 3.0 to 9.2",
                   abs((3.5 - a) / b - 7.59) < 0.01 and L.RAIN.min() <= 7.59 <= L.RAIN.max()))
    checks.append(("app challenge: a = 0.9, b = 0.34 gives a total squared miss below 0.45 (0.380)", abs(L.sse(0.9, 0.34, L.RAIN, L.YIELD) - 0.380) < 5e-4))
    checks.append(("bowl view: the lowest total miss over slopes is 0.378 at slope 0.344", abs(L.sse_for_slope(b, L.RAIN, L.YIELD) - 0.378) < 5e-4))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
